In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\dev\projects\fourlang_translation")

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "small100"
RESULT_DIR = PROJECT_ROOT / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("RESULT_DIR:", RESULT_DIR)

PROJECT_ROOT: D:\dev\projects\fourlang_translation
MODEL_DIR: D:\dev\projects\fourlang_translation\models\small100
RESULT_DIR: D:\dev\projects\fourlang_translation\results


In [2]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Current device:", device)

Python: 3.14.6 (main, Jun 23 2026, 15:20:04) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.13.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA: 13.2
Current device: cuda


In [3]:
from huggingface_hub import snapshot_download

MODEL_NAME = "alirezamsh/small100"

snapshot_download(
    repo_id=MODEL_NAME,
    local_dir=str(MODEL_DIR),
    allow_patterns=[
        "config.json",
        "generation_config.json",
        "model.safetensors",
        "sentencepiece.bpe.model",
        "special_tokens_map.json",
        "tokenizer_config.json",
        "vocab.json",
        "tokenization_small100.py",
    ],
)

print("Model saved to:", MODEL_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Model saved to: D:\dev\projects\fourlang_translation\models\small100


In [4]:
from pathlib import Path

tokenizer_file = MODEL_DIR / "tokenization_small100.py"

text = tokenizer_file.read_text(encoding="utf-8")

old = "from transformers.tokenization_utils import BatchEncoding, PreTrainedTokenizer"

new = """from transformers.tokenization_utils_base import BatchEncoding
from transformers.tokenization_utils import PreTrainedTokenizer"""

if old in text:
    text = text.replace(old, new)
    tokenizer_file.write_text(text, encoding="utf-8")
    print("tokenization_small100.py 修复成功")
else:
    print("未找到旧 import，可能已经修复过")

print(tokenizer_file)

tokenization_small100.py 修复成功
D:\dev\projects\fourlang_translation\models\small100\tokenization_small100.py


In [5]:
import sys

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from tokenization_small100 import SMALL100Tokenizer

tokenizer = SMALL100Tokenizer.from_pretrained(
    str(MODEL_DIR)
)

print("Tokenizer loaded.")

Tokenizer loaded.


In [6]:
from transformers import M2M100ForConditionalGeneration

model = M2M100ForConditionalGeneration.from_pretrained(
    str(MODEL_DIR)
)

model = model.to(device)
model.eval()

print("Model loaded.")

Loading weights:   0%|          | 0/275 [00:00<?, ?it/s]

Model loaded.


In [7]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Parameters: {num_params:,}")
print(f"Parameters: {num_params / 1e6:.2f} M")

Parameters: 332,735,488
Parameters: 332.74 M


In [8]:
import time
import torch

@torch.inference_mode()
def translate(text, target_lang):

    tokenizer.tgt_lang = target_lang

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    output = model.generate(
        **inputs,
        num_beams=5,
        max_new_tokens=128,
        early_stopping=True,
    )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    result = tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
    )[0]

    return result, elapsed

In [9]:
text = "我明天早上八点去机场。"

result, latency = translate(
    text,
    "en"
)

print("原文:", text)
print("译文:", result)
print("耗时:", latency)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


原文: 我明天早上八点去机场。
译文: I will go to the airport tomorrow at 8am.
耗时: 0.9671133999945596


In [10]:
text = "我明天早上八点去机场。"

result, latency = translate(
    text,
    "uz"
)

print("原文:", text)
print("乌兹别克语:", result)
print("耗时:", latency)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


原文: 我明天早上八点去机场。
乌兹别克语: Sabah 8 daqiqada aeroportga qaytdim.
耗时: 0.2139962000073865


In [11]:
TEST_SENTENCES = {
    "zh": "我明天早上八点去机场。",
    "en": "I will go to the airport at eight tomorrow morning.",
    "ru": "Завтра утром в восемь я поеду в аэропорт.",
    "uz": "Men ertaga ertalab soat sakkizda aeroportga boraman.",
}

In [12]:
results = []

for src_lang, source_text in TEST_SENTENCES.items():

    for tgt_lang in TEST_SENTENCES:

        if src_lang == tgt_lang:
            continue

        translation, latency = translate(
            source_text,
            tgt_lang
        )

        item = {
            "src_lang": src_lang,
            "tgt_lang": tgt_lang,
            "source_text": source_text,
            "translation": translation,
            "latency_seconds": round(latency, 4),
        }

        results.append(item)

        print("=" * 70)
        print(f"{src_lang} -> {tgt_lang}")
        print("Source:", source_text)
        print("Translation:", translation)
        print("Latency:", round(latency, 4))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

zh -> en
Source: 我明天早上八点去机场。
Translation: I will go to the airport tomorrow at 8am.
Latency: 0.1681
zh -> ru
Source: 我明天早上八点去机场。
Translation: Завтра утром в восемь поедем в аэропорт.
Latency: 0.0911
zh -> uz
Source: 我明天早上八点去机场。
Translation: Sabah 8 daqiqada aeroportga qaytdim.
Latency: 0.0676


[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


en -> zh
Source: I will go to the airport at eight tomorrow morning.
Translation: 我明天早上8点上机场。
Latency: 0.0674
en -> ru
Source: I will go to the airport at eight tomorrow morning.
Translation: В аэропорт поедем завтра в восемь утра.
Latency: 0.0724
en -> uz
Source: I will go to the airport at eight tomorrow morning.
Translation: Sabah 8 daqiqada aeroportga qaytdim.
Latency: 0.0609


[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ru -> zh
Source: Завтра утром в восемь я поеду в аэропорт.
Translation: 明天早上8点,我要去机场。
Latency: 0.0698
ru -> en
Source: Завтра утром в восемь я поеду в аэропорт.
Translation: Tomorrow at 8 am I will go to the airport.
Latency: 0.0775
ru -> uz
Source: Завтра утром в восемь я поеду в аэропорт.
Translation: Sabah 8 daqiqada aeroportga qaytdim.
Latency: 0.0673


[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


uz -> zh
Source: Men ertaga ertalab soat sakkizda aeroportga boraman.
Translation: 但是,他还会在空港上发出一声声。
Latency: 0.0874
uz -> en
Source: Men ertaga ertalab soat sakkizda aeroportga boraman.
Translation: But it's a good time to get to the airport.
Latency: 0.0849
uz -> ru
Source: Men ertaga ertalab soat sakkizda aeroportga boraman.
Translation: Вместе с тем, в аэропорту «Бордан».
Latency: 0.0979


In [13]:
len(results)

12

In [15]:
import pandas as pd

df = pd.DataFrame(results)

df

,src_lang,tgt_lang,source_text,translation,latency_seconds
0,zh,en,我明天早上八点去机场。,I will go to the airport tomorrow at 8am.,0.1681
1,zh,ru,我明天早上八点去机场。,Завтра утром в восемь поедем в аэропорт.,0.0911
2,zh,uz,我明天早上八点去机场。,Sabah 8 daqiqada aeroportga qaytdim.,0.0676
3,en,zh,I will go to the airport at eight tomorrow mor...,我明天早上8点上机场。,0.0674
4,en,ru,I will go to the airport at eight tomorrow mor...,В аэропорт поедем завтра в восемь утра.,0.0724
5,en,uz,I will go to the airport at eight tomorrow mor...,Sabah 8 daqiqada aeroportga qaytdim.,0.0609
6,ru,zh,Завтра утром в восемь я поеду в аэропорт.,"明天早上8点,我要去机场。",0.0698
7,ru,en,Завтра утром в восемь я поеду в аэропорт.,Tomorrow at 8 am I will go to the airport.,0.0775
8,ru,uz,Завтра утром в восемь я поеду в аэропорт.,Sabah 8 daqiqada aeroportga qaytdim.,0.0673
9,uz,zh,Men ertaga ertalab soat sakkizda aeroportga bo...,"但是,他还会在空港上发出一声声。",0.0874


In [16]:
UZ_TESTS = [
    "Salom.",
    "Rahmat.",
    "Men talabaman.",
    "Bugun havo yaxshi.",
    "Men Toshkentda yashayman.",
    "Men ertaga ishga boraman.",
    "Men ertaga aeroportga boraman.",
    "Men ertaga ertalab soat sakkizda aeroportga boraman.",
]

In [17]:
uz_results = []

for text in UZ_TESTS:

    translation, latency = translate(
        text,
        "en"
    )

    uz_results.append({
        "source": text,
        "translation": translation,
        "latency": latency
    })

import pandas as pd

pd.DataFrame(uz_results)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,source,translation,latency
0,Salom.,and Salom.,0.140452
1,Rahmat.,and mercy.,0.027255
2,Men talabaman.,But toughness.,0.031256
3,Bugun havo yaxshi.,by Sa'ad ibn abi Waqqas,0.049706
4,Men Toshkentda yashayman.,But in the shadow.,0.047377
5,Men ertaga ishga boraman.,But I think Issga Boraman.,0.048906
6,Men ertaga aeroportga boraman.,But in the airport Boraman.,0.050222
7,Men ertaga ertalab soat sakkizda aeroportga bo...,But it's a good time to get to the airport.,0.084247


In [18]:
uz_text = (
    "Men ertaga ertalab soat sakkizda "
    "aeroportga boraman."
)

tokenizer.tgt_lang = "en"

encoded = tokenizer(
    uz_text,
    return_tensors="pt"
)

tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

print("原文：")
print(uz_text)

print("\nTokens：")
print(tokens)

print("\nInput IDs：")
print(encoded["input_ids"][0].tolist())

原文：
Men ertaga ertalab soat sakkizda aeroportga boraman.

Tokens：
['__en__', '▁Men', '▁er', 'ta', 'ga', '▁er', 'tal', 'ab', '▁so', 'at', '▁sak', 'k', 'iz', 'da', '▁aer', 'oport', 'ga', '▁bor', 'aman', '.', '</s>']

Input IDs：
[128022, 1570, 62, 148, 467, 62, 5781, 303, 324, 59, 3187, 106, 384, 232, 18645, 48938, 467, 2590, 1905, 5, 2]


In [19]:
unk_token = tokenizer.unk_token

unk_count = tokens.count(unk_token)

print("Token 数量:", len(tokens))
print("<unk> 数量:", unk_count)
print(
    "UNK 比例:",
    unk_count / len(tokens)
)

Token 数量: 21
<unk> 数量: 0
UNK 比例: 0.0


In [20]:
SMALL100_RESULT_FILE = RESULT_DIR / "small100_12dir_baseline.csv"

df.to_csv(
    SMALL100_RESULT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", SMALL100_RESULT_FILE)

Saved: D:\dev\projects\fourlang_translation\results\small100_12dir_baseline.csv


In [21]:
import pandas as pd

small100_uz_df = pd.DataFrame(uz_results)

small100_uz_df.to_csv(
    RESULT_DIR / "small100_uz_to_en_test.csv",
    index=False,
    encoding="utf-8-sig"
)

In [22]:
import gc
import torch

try:
    del model
except NameError:
    pass

try:
    del tokenizer
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("SMaLL-100 model released.")

SMaLL-100 model released.


In [23]:
import torch
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
)

M2M_MODEL_NAME = "facebook/m2m100_418M"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [24]:
m2m_tokenizer = M2M100Tokenizer.from_pretrained(
    M2M_MODEL_NAME
)

m2m_model = M2M100ForConditionalGeneration.from_pretrained(
    M2M_MODEL_NAME
)

m2m_model = m2m_model.to(device)
m2m_model.eval()

print("M2M100-418M loaded.")

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 2.42MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

M2M100-418M loaded.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            

In [25]:
num_params = sum(
    p.numel()
    for p in m2m_model.parameters()
)

print(f"Parameters: {num_params:,}")
print(f"Parameters: {num_params / 1e6:.2f} M")

Parameters: 483,905,536
Parameters: 483.91 M


In [26]:
for lang in ["zh", "en", "ru", "uz"]:
    lang_id = m2m_tokenizer.get_lang_id(lang)
    print(
        f"{lang}:",
        lang_id
    )

zh: 128102
en: 128022
ru: 128077
uz: 128096


In [27]:
import time
import torch


@torch.inference_mode()
def translate_m2m(
    text: str,
    src_lang: str,
    tgt_lang: str,
):
    # 1. 指定源语言
    m2m_tokenizer.src_lang = src_lang

    # 2. Tokenize
    inputs = m2m_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    # GPU计时前同步
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    # 3. 指定目标语言
    generated_tokens = m2m_model.generate(
        **inputs,
        forced_bos_token_id=m2m_tokenizer.get_lang_id(
            tgt_lang
        ),
        num_beams=5,
        max_new_tokens=128,
        early_stopping=True,
    )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    # 4. Decode
    translation = m2m_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )[0]

    return translation, elapsed

In [28]:
text = "我明天早上八点去机场。"

result, latency = translate_m2m(
    text=text,
    src_lang="zh",
    tgt_lang="en",
)

print("原文:", text)
print("译文:", result)
print("耗时:", latency)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


原文: 我明天早上八点去机场。
译文: I will go to the airport tomorrow morning at 8 p.m.
耗时: 0.4691088000545278


In [29]:
@torch.inference_mode()
def translate_m2m_greedy(
    text,
    src_lang,
    tgt_lang,
):
    m2m_tokenizer.src_lang = src_lang

    inputs = m2m_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    output = m2m_model.generate(
        **inputs,
        forced_bos_token_id=m2m_tokenizer.get_lang_id(
            tgt_lang
        ),
        num_beams=1,
        do_sample=False,
        max_new_tokens=64,
    )

    return m2m_tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
    )[0]

In [30]:
text = "我明天早上八点去机场。"

print(
    translate_m2m_greedy(
        text,
        "zh",
        "en"
    )
)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I will go to the airport tomorrow morning at 8 p.m.


In [31]:
m2m_tokenizer.src_lang = "zh"

text = "我明天早上八点去机场。"

inputs = m2m_tokenizer(
    text,
    return_tensors="pt",
).to(device)

outputs = m2m_model.generate(
    **inputs,
    forced_bos_token_id=m2m_tokenizer.get_lang_id("en"),
    num_beams=5,
    num_return_sequences=5,
    max_new_tokens=64,
    early_stopping=True,
)

candidates = m2m_tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

for i, sentence in enumerate(candidates, 1):
    print(f"{i}: {sentence}")

[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1: I will go to the airport tomorrow morning at 8 p.m.
2: I will go to the airport tomorrow morning at eight.
3: I am going to the airport tomorrow morning at eight.
4: I’ll go to the airport tomorrow morning at eight.
5: I’ll go to the airport tomorrow morning at eighth.


In [33]:
UZ_TESTS = [
    "Salom.",
    "Rahmat.",
    "Men talabaman.",
    "Bugun havo yaxshi.",
    "Men Toshkentda yashayman.",
    "Men ertaga ishga boraman.",
    "Men ertaga aeroportga boraman.",
    "Men ertaga ertalab soat sakkizda aeroportga boraman.",
]

In [34]:
TIME_TESTS = [
    "我明天早上八点去机场。",
    "我明天上午八点去机场。",
    "我明天早上8点去机场。",
    "我明天上午8点去机场。",
    "我明天8:00去机场。",
    "我明天早上8:00去机场。",
]

In [35]:
time_results = []

for text in TIME_TESTS:
    translation, latency = translate_m2m(
        text=text,
        src_lang="zh",
        tgt_lang="en",
    )

    time_results.append({
        "source": text,
        "translation": translation,
        "latency": latency,
    })

import pandas as pd

pd.DataFrame(time_results)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,source,translation,latency
0,我明天早上八点去机场。,I will go to the airport tomorrow morning at 8...,0.334941
1,我明天上午八点去机场。,I will go to the airport tomorrow at 8 a.m.,0.214837
2,我明天早上8点去机场。,I will go to the airport tomorrow morning at 8...,0.162695
3,我明天上午8点去机场。,I will go to the airport tomorrow at 8 a.m.,0.160208
4,我明天8:00去机场。,I go to the airport tomorrow at 8:00 p.m.,0.161310
5,我明天早上8:00去机场。,I will go to the airport tomorrow morning at 8...,0.156875


In [36]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

In [37]:
pd.DataFrame(time_results)

,source,translation,latency
0,我明天早上八点去机场。,I will go to the airport tomorrow morning at 8 p.m.,0.334941
1,我明天上午八点去机场。,I will go to the airport tomorrow at 8 a.m.,0.214837
2,我明天早上8点去机场。,I will go to the airport tomorrow morning at 8 p.m.,0.162695
3,我明天上午8点去机场。,I will go to the airport tomorrow at 8 a.m.,0.160208
4,我明天8:00去机场。,I go to the airport tomorrow at 8:00 p.m.,0.161310
5,我明天早上8:00去机场。,I will go to the airport tomorrow morning at 8:00 p.m.,0.156875


In [38]:
UZ_TESTS = [
    "Salom.",
    "Rahmat.",
    "Men talabaman.",
    "Bugun havo yaxshi.",
    "Men Toshkentda yashayman.",
    "Men ertaga ishga boraman.",
    "Men ertaga aeroportga boraman.",
    "Men ertaga ertalab soat sakkizda aeroportga boraman.",
]

In [39]:
import pandas as pd

m2m_uz_en_results = []

for text in UZ_TESTS:
    translation, latency = translate_m2m(
        text=text,
        src_lang="uz",
        tgt_lang="en",
    )

    m2m_uz_en_results.append({
        "source": text,
        "translation": translation,
        "latency": round(latency, 4),
    })

m2m_uz_en_df = pd.DataFrame(m2m_uz_en_results)

pd.set_option("display.max_colwidth", None)

m2m_uz_en_df

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,source,translation,latency
0,Salom.,and Salom.,0.1725
1,Rahmat.,and mercy.,0.0744
2,Men talabaman.,But it is.,0.0751
3,Bugun havo yaxshi.,by Havo Nashid.,0.0849
4,Men Toshkentda yashayman.,You are in Yashima.,0.0841
5,Men ertaga ishga boraman.,But it is Issue.,0.0890
6,Men ertaga aeroportga boraman.,and the airport.,0.0813
7,Men ertaga ertalab soat sakkizda aeroportga boraman.,The airport is located on the airport.,0.1194
